# Step 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')
pd.set_option('display.max_columns', None)

print('done')

# Step 2 — Load the data

In [ ]:
df = pd.read_csv('data.csv')

print('Rows   :', df.shape[0])
print('Columns:', df.shape[1])

# Step 3 — Look at the actual rows

In [ ]:
df.head(10)

# Step 4 — Column types and missing values

In [ ]:
df.info()

# Step 5 — Summary statistics

In [ ]:
df.describe().T

# Observations from describe()

| Column | Issue | Action |
|---|---|---|
| `Annual_Premium` | max=540K, 75%=39K — extreme outlier | Cap before scaling |
| `Response` | mean=0.12 — 12% positive only | Use ROC-AUC, not accuracy |
| `Driving_License` | mean=0.997 — no variation | Low value feature |
| `id` | row number | Drop before modeling |

# Step 6 — Unique values in text columns

In [ ]:
for col in ['Gender', 'Vehicle_Age', 'Vehicle_Damage']:
    print(f'{col}: {df[col].unique()}')
    print(f'  counts:\n{df[col].value_counts().to_string()}')
    print()

# Step 7 — Target column (Response)

In [ ]:
counts = df['Response'].value_counts()
pct    = df['Response'].value_counts(normalize=True) * 100

print('Count:')
print(counts.to_string())
print('\nPercentage:')
print(pct.round(1).to_string())

plt.figure(figsize=(5, 4))
counts.plot(kind='bar', color=['#e74c3c', '#2ecc71'])
plt.xticks([0, 1], ['Not Interested (0)', 'Interested (1)'], rotation=0)
plt.title('Target Column — Response')
plt.ylabel('Number of customers')
plt.tight_layout()
plt.show()

In [ ]:
# Fix column name that still has special character
df_model = df_model.rename(columns={'Vehicle_Age_< 1 Year': 'Vehicle_Age_lt_1_Year'})

# Convert bool columns to int (True/False → 1/0)
df_model['Vehicle_Age_lt_1_Year']   = df_model['Vehicle_Age_lt_1_Year'].astype(int)
df_model['Vehicle_Age_gt_2_Years']  = df_model['Vehicle_Age_gt_2_Years'].astype(int)

print('Columns:', df_model.columns.tolist())
df_model.head(3)

## Step 10 — Split into Train and Test\nSplit BEFORE scaling. Test data must never influence the scaler.

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model.drop('Response', axis=1)   # features
y = df_model['Response']                 # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,    # 75% train, 25% test
    random_state=42,   # same split every time you run
    stratify=y         # keeps same 88/12 ratio in both splits
)

print(f'Train : {X_train.shape[0]:,} rows')
print(f'Test  : {X_test.shape[0]:,} rows')
print(f'\nTrain positive class: {y_train.mean()*100:.1f}%')
print(f'Test  positive class: {y_test.mean()*100:.1f}%')

## Step 11 — Scale numeric columns\nFit scaler on train only, then apply to both train and test.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# StandardScaler — for Age and Vintage (roughly normal distributions)
# Converts to mean=0, std=1
ss = StandardScaler()
X_train[['Age', 'Vintage']] = ss.fit_transform(X_train[['Age', 'Vintage']])
X_test[['Age', 'Vintage']]  = ss.transform(X_test[['Age', 'Vintage']])

# MinMaxScaler — for Annual_Premium (has extreme outliers, squash to 0-1 range)
mm = MinMaxScaler()
X_train[['Annual_Premium']] = mm.fit_transform(X_train[['Annual_Premium']])
X_test[['Annual_Premium']]  = mm.transform(X_test[['Annual_Premium']])

print('Before scaling — Annual_Premium range in test:')
print(f'  min: {X_test["Annual_Premium"].min():.4f}')
print(f'  max: {X_test["Annual_Premium"].max():.4f}')
print(f'\nBefore scaling — Age range in test:')
print(f'  min: {X_test["Age"].min():.2f}')
print(f'  max: {X_test["Age"].max():.2f}')
print('\nScaling done.')

## Step 12 — Train a Random Forest model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,      # 100 decision trees
    max_depth=10,          # each tree can go 10 levels deep
    class_weight='balanced', # handles 88/12 imbalance
    random_state=42,
    n_jobs=-1              # use all CPU cores
)

rf.fit(X_train, y_train)
print('Model trained.')

## Step 13 — Evaluate the model

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]  # probability of being class 1

print(classification_report(y_test, y_pred, target_names=['Not Interested', 'Interested']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}')

## Step 14 — Feature Importance\nDoes the model agree with what we found in EDA?

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
feat_imp.plot(kind='bar', color='steelblue')
plt.title('Feature Importance — what the model actually learned')
plt.ylabel('Importance score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(feat_imp.round(3))

## Finding the right max_depth — train score vs test score

In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths       = [2, 3, 4, 5, 6, 8, 10, 15, 20]
train_scores = []
test_scores  = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, class_weight='balanced', random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))

# Plot
plt.figure(figsize=(9, 5))
plt.plot(depths, train_scores, marker='o', label='Train score', color='#2ecc71')
plt.plot(depths, test_scores,  marker='o', label='Test score',  color='#e74c3c')
plt.title('Train vs Test Score at Different Depths')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.legend()
plt.xticks(depths)
plt.tight_layout()
plt.show()

# Print the numbers
print(f'{"Depth":<8} {"Train":>8} {"Test":>8} {"Gap":>8}')
print('-' * 35)
for d, tr, te in zip(depths, train_scores, test_scores):
    print(f'{d:<8} {tr:>8.3f} {te:>8.3f} {tr-te:>8.3f}')

## Model 1 — Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

lr = LogisticRegression(
    C=1.0,                   # default regularization
    max_iter=1000,           # enough steps to converge
    class_weight='balanced', # handles 88/12 imbalance
    random_state=42
)

lr.fit(X_train, y_train)

y_pred  = lr.predict(X_test)
y_proba = lr.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Not Interested', 'Interested']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}')
print(f'\nRandom Forest was : 0.8544  ← compare against this')

In [ ]:
# Logistic Regression shows exact weight for each feature
# Positive weight → pushes toward Interested
# Negative weight → pushes toward Not Interested

coefficients = pd.Series(lr.coef_[0], index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
coefficients.plot(kind='bar', color=['#2ecc71' if v > 0 else '#e74c3c' for v in coefficients])
plt.title('Logistic Regression — Feature Coefficients')
plt.ylabel('Weight')
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(coefficients.round(3))

## Model 2 — Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    max_depth=5,             # from our depth experiment, sweet spot was 3-5
    min_samples_leaf=20,     # at least 20 customers at each leaf
    class_weight='balanced',
    random_state=42
)

dt.fit(X_train, y_train)

y_pred  = dt.predict(X_test)
y_proba = dt.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Not Interested', 'Interested']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}')
print()
print('--- Scoreboard ---')
print(f'Logistic Regression : 0.8378')
print(f'Decision Tree       : {roc_auc_score(y_test, y_proba):.4f}')
print(f'Random Forest       : 0.8544')

In [ ]:
from sklearn.tree import export_text

# Print the actual questions the tree learned
tree_rules = export_text(dt, feature_names=list(X_train.columns), max_depth=3)
print(tree_rules)

## Model 3 — XGBoost

In [ ]:
# XGBoost requires all columns to be numeric
# Convert any remaining object columns to float
X_train = X_train.astype(float)
X_test  = X_test.astype(float)

print('Dtypes after fix:')
print(X_train.dtypes)

In [ ]:
from xgboost import XGBClassifier

# class imbalance handled via scale_pos_weight
# = number of negatives / number of positives
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f'scale_pos_weight = {scale:.1f}  (handles 88/12 imbalance)')

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    scale_pos_weight=scale,   # XGBoost's version of class_weight='balanced'
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    verbosity=0
)

xgb.fit(X_train, y_train)

y_pred  = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['Not Interested', 'Interested']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}')
print()
print('--- Scoreboard ---')
print(f'Logistic Regression : 0.8378')
print(f'Decision Tree       : 0.8378')
print(f'Random Forest       : 0.8544')
print(f'XGBoost             : {roc_auc_score(y_test, y_proba):.4f}')

In [ ]:
feat_imp = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
feat_imp.plot(kind='bar', color='steelblue')
plt.title('XGBoost — Feature Importance')
plt.ylabel('Importance score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(feat_imp.round(3))

## SMOTE — does it improve over built-in imbalance handling?

In [ ]:
from sklearn.metrics import recall_score

In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE on training data only — never on test
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print('Before SMOTE:')
print(f'  NO  : {(y_train==0).sum():,}')
print(f'  YES : {(y_train==1).sum():,}')
print(f'\nAfter SMOTE:')
print(f'  NO  : {(y_train_sm==0).sum():,}')
print(f'  YES : {(y_train_sm==1).sum():,}')

In [ ]:
# Train XGBoost on SMOTE data — no scale_pos_weight needed, data is already balanced
xgb_smote = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    verbosity=0
)

xgb_smote.fit(X_train_sm, y_train_sm)

y_pred_sm  = xgb_smote.predict(X_test)
y_proba_sm = xgb_smote.predict_proba(X_test)[:, 1]

print('=== XGBoost WITH SMOTE ===')
print(classification_report(y_test, y_pred_sm, target_names=['Not Interested', 'Interested']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba_sm):.4f}')

print()
print('--- Final Comparison ---')
print(f'XGBoost without SMOTE : 0.8578  recall={0.93:.2f}')
print(f'XGBoost with SMOTE    : {roc_auc_score(y_test, y_proba_sm):.4f}  recall={recall_score(y_test, y_pred_sm):.2f}')

## Hyperparameter Tuning — XGBoost with RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Parameter grid — ranges to search through
param_grid = {
    'n_estimators'  : [100, 200, 300, 500],
    'learning_rate' : [0.01, 0.05, 0.1, 0.2],
    'max_depth'     : [3, 4, 5, 6],
    'subsample'     : [0.7, 0.8, 0.9, 1.0],   # fraction of rows each tree sees
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]  # fraction of features each tree sees
}

base_xgb = XGBClassifier(
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    verbosity=0
)

search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_grid,
    n_iter=20,          # try 20 random combinations
    scoring='roc_auc',  # optimise for ROC-AUC
    cv=3,               # 3-fold cross validation for each combination
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print('Starting search — trying 20 combinations with 3-fold CV each...')
print('This will take 2-5 minutes.')
search.fit(X_train, y_train)

In [ ]:
print('Best parameters found:')
for param, value in search.best_params_.items():
    print(f'  {param:20s} : {value}')

print(f'\nBest CV ROC-AUC : {search.best_score_:.4f}')

# Evaluate best model on test set
best_model  = search.best_estimator_
y_pred_tuned  = best_model.predict(X_test)
y_proba_tuned = best_model.predict_proba(X_test)[:, 1]

print(f'Test  ROC-AUC   : {roc_auc_score(y_test, y_proba_tuned):.4f}')

print()
print('--- Before vs After Tuning ---')
print(f'XGBoost default : 0.8578   recall=0.93')
print(f'XGBoost tuned   : {roc_auc_score(y_test, y_proba_tuned):.4f}   recall={recall_score(y_test, y_pred_tuned):.2f}')

# Step 8 — Bivariate Analysis (each feature vs target)

In [ ]:
# Previously Insured
rate = df.groupby('Previously_Insured')['Response'].mean() * 100
print('Previously Insured:')
print(f'  Not insured (0) : {rate[0]:.1f}%')
print(f'  Insured     (1) : {rate[1]:.1f}%')

In [ ]:
# Vehicle Damage
rate = df.groupby('Vehicle_Damage')['Response'].mean() * 100
print('Vehicle Damage:')
for group, r in rate.items():
    print(f'  {group} : {r:.1f}%')

In [ ]:
# Vehicle Age
rate = df.groupby('Vehicle_Age')['Response'].mean() * 100
rate = rate.reindex(['< 1 Year', '1-2 Year', '> 2 Years'])
print('Vehicle Age:')
for group, r in rate.items():
    print(f'  {group} : {r:.1f}%')

In [ ]:
# Age
print('Mean Age — Interested    :', df[df['Response']==1]['Age'].mean().round(1))
print('Mean Age — Not Interested:', df[df['Response']==0]['Age'].mean().round(1))

In [ ]:
# Vintage
print('Mean Vintage — Interested    :', df[df['Response']==1]['Vintage'].mean().round(1))
print('Mean Vintage — Not Interested:', df[df['Response']==0]['Vintage'].mean().round(1))

# Preprocessing
Order matters:
1. Drop id
2. Encode text columns
3. Split into train/test
4. Scale numeric columns

## Step 9 — Drop id, encode all text columns

In [ ]:
df_model = df.copy()

# Drop id
df_model = df_model.drop('id', axis=1)

# Label encode — 2-value columns
df_model['Gender']         = df_model['Gender'].replace({'Male': 1, 'Female': 0})
df_model['Vehicle_Damage'] = df_model['Vehicle_Damage'].replace({'Yes': 1, 'No': 0})

# One-hot encode — Vehicle_Age (3 categories)
df_model = pd.get_dummies(df_model, columns=['Vehicle_Age'], drop_first=True)

# Clean up column names
df_model.columns = df_model.columns.str.replace('< ', 'lt_').str.replace('> ', 'gt_').str.replace(' ', '_').str.replace('-', '_')

print('Columns:', df_model.columns.tolist())
print('Shape  :', df_model.shape)
df_model.head(3)